# Data Management Exercise (Python)

In this exercise, you will:

- Inspect datasets
- Apply minimal cleaning
- Merge datasets
- Create an analysis-ready dataset

Focus: understanding the workflow, not programming.

## How to run the notebook

- Click into a cell to activate it  
- Run a cell: **Shift + Enter**  
- Work step by step — each cell builds on the previous one
- If something fails: restart from the top and run again  

# Step 1: Load the data

We start by loading the raw datasets.

- `participants_exercise.csv`: basic participant information  

In [ ]:
# =============================================
# Read, inspect and clean the participants data
# =============================================

# Import the pandas package which allows to work with structured data tables in Python
import pandas as pd

# Read the participants data into the dataframe "dfp"
dfp=pd.read_csv("participants_exercise.csv", dtype={"id": str}) # Keep id data type as string to preserve leading zeros

## Inspect the dataset

First look at the structure:

- What variables are available?
- What does one row represent?

In [ ]:
# Show the head of the dataset and inspect the structure like column names, date format, etc
dfp.head()

## Check data types

- Are variables stored as expected?
- Are dates, numbers, and text correctly interpreted?
- Are there variables with missing data?

In [ ]:
# Check data types
dfp.info()

## Explore a variable

Look at the distribution of values.

Questions:

- Are categories consistent?
- Do you see unexpected values?

In [ ]:
# Inspect the variable sex
dfp["sex"].value_counts(dropna=False)

In [ ]:
# Standardize the variable sex
dfp["sex"] = dfp["sex"].replace({
    "Female": "F"
})

In [ ]:
# Check if cleaning was successful
dfp["sex"].value_counts(dropna=False)

## Handling inconsistent categories

Real datasets often contain:

- different spellings
- mixed capitalization
- combined categories

In [ ]:
# Inspect the variable diagnosis
dfp["diagnosis"].value_counts(dropna=False).sort_index()

In [ ]:
# Standardize Diabetes
dfp["diagnosis"]=dfp["diagnosis"].replace({
    "diabetes": "Diabetes",
    "Diabtes": "Diabetes",
    "DM": "Diabetes"
})

In [ ]:
# Check if cleaning was successful
dfp["diagnosis"].value_counts(dropna=False).sort_index()

## Date formats and data types

The variable `birth_date` looks like a date (YYYY-MM-DD),  
but is currently stored as **text (string)**.

For analysis, we need a proper **datetime format**.

This allows:
- calculations (e.g., age)
- correct sorting and filtering

We convert the variable to a datetime type in the next step.

In [ ]:
# birth_date has the correct format YYYY-MM-DD, but is internally treated as string
# Convert it to datetime data type (we need that later)
dfp['birth_date'] = pd.to_datetime(dfp['birth_date'], format='%Y-%m-%d')

## Step 2: Load second dataset

Now we load the visits dataset.

In [ ]:
# =============================================
# Read, inspect and clean the visits data
# =============================================

dfv=pd.read_excel("visits_exercise.xlsx", dtype={"id": str})
dfv.head()

## Date formats

Dates often appear in different formats.

Here:
- visit_date: "DD.MM.YYYY"

We convert to a standard format for analysis: "YYYY-MM-DD"

In [ ]:
# Convert visit_date to datetime data type
dfv['visit_date'] = pd.to_datetime(dfv['visit_date'], format='%d.%m.%Y')

# Format as yyyy-mm-dd for representation
#dfv['visit_date'] = dfv['visit_date'].dt.strftime('%Y-%m-%d')

dfv.head()

## Step 3: Merge datasets

We combine:

- participant information
- visit data

Using the key variable: **id**

In [ ]:
# =============================================
# Merge participants and visits data
# =============================================

df_result=dfp.merge(dfv, on="id", how="left")
df_result.head()

### Discussion

- What happens if an ID exists in one dataset but not the other?
- Why was join type "left" used here? How does it work?

## Step 4: Create derived variables

Derived variables are created during data processing,
not stored in raw data.

We calculate age at the time of visit.

In [ ]:
# Calculate age at visit_date
df_result["age"]=(df_result["visit_date"] - df_result["birth_date"]).dt.days / 365.25
df_result.head()

## Step 5: Save analysis-ready data

We export the cleaned and merged dataset.

This dataset is now ready for analysis.

In [ ]:
# =============================================
# Save result
# =============================================

# Please observe:
# Neither participants nor visits data has been cleaned in all aspects due to time limitations of this exercise
# However, let's assume df_result is the cleaned final version
# Save it in CSV format
# In this Binder environment, you find the saved file in the "View" menu -> "File browser"

df_result.to_csv("participants_visits_20260324.csv", index=False)


## Summary

You performed the key steps of a data workflow:

- Inspect data
- Clean variables
- Standardize formats
- Merge datasets
- Create derived variables

These steps are the same across tools:

- OpenRefine → interactive cleaning  
- Python / R → scripted, reproducible workflows  

## Optional Section (if you have time)

The following steps extend the workflow and show additional data management tasks.

This part is optional and meant to give a more complete picture.  
Focus on understanding the workflow rather than finishing everything.

You can explore:
- further data checks and validation
- additional transformations
- completing the full pipeline

In [ ]:
# =============================================
# O P T I O N A L
# If you have time
# =============================================

# Check duplicated ids in participants data
dfp[dfp["id"].duplicated(keep=False)].sort_values("id")

In [ ]:
# When you inspect the duplicated records above
# you'll see that except for id 004 all other records are exact duplicates of all fields values

# In larger datasets visual inspection is not applicable
# Check duplicated values across all fields (=exact copies of a record, not only duplicated id)
dfp[dfp.duplicated(keep=False)].sort_values("id")

In [ ]:
# Remove duplicates with identical values in all fields
dfp=dfp.drop_duplicates()

# Check result
dfp[dfp.duplicated(keep=False)].sort_values("id")

In [ ]:
# Back to the record with only duplicated id, but different values in other fields
dfp[dfp["id"].duplicated(keep=False)].sort_values("id")

In [ ]:
# Let's assume you can go back to your original inclusion records
# 004 | F | 1988-01-30 is the correct participant
# 004 | M | 1982-11-02 was never assigned

# Remove the wrong record
dfp = dfp[~(
        (dfp["id"] == "004") &
        (dfp["sex"] == "M") &
        (dfp["birth_date"] == "1982-11-02")
    )
]

# Check result
dfp[dfp["id"].duplicated(keep=False)].sort_values("id")

In [ ]:
# Inspect cholesterol in the visits dataset
# Show the distribution

dfv["cholesterol"].describe()

In [ ]:
# Plot a histogram

dfv["cholesterol"].plot(kind="hist")

In [ ]:
# Anything to clean here?
# Inspect values > 10

dfv[dfv["cholesterol"] > 10]

In [ ]:
# Merge (again) participants and visits data

df_result = dfp.merge(dfv, on="id", how="left")
df_result.head()

In [ ]:
# Are there participants without visit?

df_result[df_result["visit_date"].isnull()]

In [ ]:
# Calculate (again) age at visit_date

df_result["age"]=(df_result["visit_date"] - df_result["birth_date"]).dt.days / 365.25
df_result.head()

In [ ]:
# Distribution of age

df_result["age"].describe()

In [ ]:
# Show records with age <= 0

df_result[df_result["age"] <= 0]

In [ ]:
# Let's assume you can go back to the original visit records and get the correct visit_date
# Set the correct visit_date

df_result.loc[
    df_result["visit_date"] == pd.Timestamp("1980-01-01"),
    "visit_date"
] = pd.Timestamp("2025-10-10")

# Check result
df_result[df_result["id"]=="004"]

In [ ]:
# Recalculate age

df_result["age"]=(df_result["visit_date"] - df_result["birth_date"]).dt.days / 365.25
df_result["age"].describe()

In [ ]:
# Save result

df_result.to_csv("participants_visits_20260324.csv", index=False)